In [ ]:
# Import Data and Setup
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
from sklearn.decomposition import LatentDirichletAllocation
!pip install pyLDAvis
import pyLDAvis
import numpy as np
import warnings
warnings.simplefilter("ignore", DeprecationWarning) # This code is extremely spammy otherwise

file_path = "/content/drive/MyDrive/clash_royale_reviews.csv"

df = pd.read_csv(
    file_path,
    usecols=["review_id", "content", "score", "at", "likes", "appVersion"],
    parse_dates=["at"]
)

In [ ]:
# Categorize Reviews Based On Score
conditions = [
(df['score'] > 2),
(df['score'] < 3)
]
choices = [
'positive',
'negative'
]
df['review_category'] = np.select(conditions, choices, default = '')

In [ ]:
#sample smaller portions of reviews to avoid crashing during analysis
sampledf = df.sample(n= 500000, random_state = 123)
smaller_sampledf = df.sample(n = 100000, random_state = 123)
vectorizer = CountVectorizer(min_df=0.005, max_df=0.9, stop_words='english')
clash_dtm = vectorizer.fit_transform(sampledf['content'].values.astype('U'))

In [ ]:
lda = LatentDirichletAllocation(n_components=20, random_state=123)
lda.fit(clash_dtm)

LatentDirichletAllocation(n_components=20, random_state=123)

In [ ]:
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_):
    top_features_ind = topic.argsort()[:-10 - 1:-1]
    top_features = [feature_names[i] for i in top_features_ind]
    print(f"Topic #{topic_idx + 1}: {', '.join(top_features)}")

Topic #1: supercell, cool, game, gameplay, know, enjoy, don, graphics, say, guys
Topic #2: fun, game, worst, interesting, trash, lot, play, used, don, make
Topic #3: like, really, game, cards, updates, making, don, lot, make, new
Topic #4: time, chest, open, chests, long, better, way, takes, stars, lot
Topic #5: win, level, game, pay, cards, players, match, higher, matchmaking, battle
Topic #6: amazing, game, lvl, friends, pretty, 10, connection, lose, easy, say
Topic #7: play, game, graphics, time, pass, good, free, easy, day, want
Topic #8: bad, hard, game, work, dont, boring, thing, gets, little, free
Topic #9: just, playing, game, app, years, awsome, stop, ve, day, enjoy
Topic #10: game, nice, addictive, enjoy, lot, play, easy, stars, graphics, battle
Topic #11: super, game, thanks, star, royal, guys, needs, clash, make, lot
Topic #12: update, new, game, money, loved, cards, want, spend, im, don
Topic #13: best, game, played, world, games, wow, mobile, ve, phone, years
Topic #14: g

In [ ]:
doctopic = lda.transform(clash_dtm)

In [ ]:
vis_data = pyLDAvis.prepare(topic_term_dists = lda.components_, # topic-word matrix
                            doc_topic_dists = doctopic, # document-topic matrix
                            doc_lengths = np.array(clash_dtm.sum(axis = 1)).flatten(), # The package expects flat arrays, so we need to convert to numpy array first and then flatten
                            vocab = vectorizer.get_feature_names_out(),
                            term_frequency = np.array(clash_dtm.sum(axis = 0)).flatten(),
                            sort_topics = True) # Set this to True to sort topics by size

In [ ]:
pyLDAvis.display(vis_data)

In [40]:
#topic models for only positive reviews
positive_df = df[df['review_category'] == 'positive']
pos_sample = positive_df.sample(n=250000, random_state=123)
vectorizer = CountVectorizer(min_df=100, max_df=0.9, stop_words='english')
positive_clash_dtm = vectorizer.fit_transform(pos_sample['content'].values.astype('U'))

In [41]:
pos_lda = LatentDirichletAllocation(n_components=20, random_state=123)
pos_lda.fit(positive_clash_dtm)

LatentDirichletAllocation(n_components=20, random_state=123)

In [42]:
pos_feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(pos_lda.components_):
    top_features_ind = topic.argsort()[:-10 - 1:-1]
    top_features = [pos_feature_names[i] for i in top_features_ind]
    print(f"Topic #{topic_idx + 1}: {', '.join(top_features)}")

Topic #1: great, game, play, fun, easy, addicted, friends, simple, overall, just
Topic #2: graphics, game, gameplay, little, make, enjoy, bit, strategic, mind, alot
Topic #3: game, best, fun, mobile, life, wonderful, far, seen, think, hi
Topic #4: game, new, strategy, download, interesting, pretty, updates, epic, cards, makes
Topic #5: nice, game, entertaining, hai, peak, yo, decent, gane, stress, try
Topic #6: level, people, game, cards, app, hard, players, lose, trophies, higher
Topic #7: love, game, playing, years, challenging, just, came, ve, concept, year
Topic #8: clash, clans, better, royale, coc, wow, like, royal, game, happy
Topic #9: legendary, card, game, cards, got, arena, fantastic, plz, pls, want
Topic #10: like, game, best, played, games, world, excellent, ve, coc, online
Topic #11: really, time, fun, game, killer, pass, lot, real, way, lots
Topic #12: good, game, pretty, job, dope, realy, overall, think, far, haha
Topic #13: loved, need, game, troops, just, player, don,

In [ ]:
pos_doctopic = pos_lda.transform(positive_clash_dtm)

In [ ]:
pos_vis_data = pyLDAvis.prepare(topic_term_dists = pos_lda.components_, # topic-word matrix
                            doc_topic_dists = pos_doctopic, # document-topic matrix
                            doc_lengths = np.array(positive_clash_dtm.sum(axis = 1)).flatten(), # The package expects flat arrays, so we need to convert to numpy array first and then flatten
                            vocab = vectorizer.get_feature_names_out(),
                            term_frequency = np.array(positive_clash_dtm.sum(axis = 0)).flatten(),
                            sort_topics = True) # Set this to True to sort topics by size

In [ ]:
pyLDAvis.display(pos_vis_data)

In [43]:
#topic models for only negative reviews
negative_df = df[df['review_category'] == 'negative']
neg_sample = negative_df.sample(n=250000, random_state=123)
vectorizer = CountVectorizer(min_df=100, max_df=0.9, stop_words='english')
negative_clash_dtm = vectorizer.fit_transform(neg_sample['content'].values.astype('U'))

In [44]:
neg_lda = LatentDirichletAllocation(n_components=20, random_state=123)
neg_lda.fit(negative_clash_dtm)

LatentDirichletAllocation(n_components=20, random_state=123)

In [45]:
neg_feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(neg_lda.components_):
    top_features_ind = topic.argsort()[:-10 - 1:-1]
    top_features = [neg_feature_names[i] for i in top_features_ind]
    print(f"Topic #{topic_idx + 1}: {', '.join(top_features)}")

Topic #1: good, que, ok, el, juego, es, la, se, man, en
Topic #2: game, connection, download, best, problem, internet, lost, network, wifi, issue
Topic #3: account, supercell, nice, super, help, star, support, nerf, game, lost
Topic #4: game, update, money, new, players, ruined, updates, supercell, years, play
Topic #5: money, game, want, people, spend, don, gems, buy, make, just
Topic #6: game, playing, just, years, ve, play, time, fun, day, played
Topic #7: bad, chest, game, open, time, chests, long, takes, wait, hours
Topic #8: match, fix, game, battle, making, lose, trophies, time, crashes, lost
Topic #9: cards, arena, game, people, players, higher, card, just, horrible, levels
Topic #10: game, fun, just, used, great, worse, garbage, good, anymore, skill
Topic #11: clash, royale, pass, clans, royal, like, game, wow, rubbish, crash
Topic #12: level, lvl, cards, 10, higher, 13, game, 12, 11, players
Topic #13: update, game, clan, play, app, 50, new, let, fix, open
Topic #14: deck, ga

In [32]:
neg_doctopic = neg_lda.transform(negative_clash_dtm)

In [30]:
neg_vis_data = pyLDAvis.prepare(topic_term_dists = neg_lda.components_, # topic-word matrix
                            doc_topic_dists = neg_doctopic, # document-topic matrix
                            doc_lengths = np.array(negative_clash_dtm.sum(axis = 1)).flatten(), # The package expects flat arrays, so we need to convert to numpy array first and then flatten
                            vocab = vectorizer.get_feature_names_out(),
                            term_frequency = np.array(negative_clash_dtm.sum(axis = 0)).flatten(),
                            sort_topics = True) # Set this to True to sort topics by size

In [34]:
pyLDAvis.display(neg_vis_data)